In [1]:
# Cell 1: TF-IDF setup used by every text model
from pathlib import Path

import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
OUTPUT_DIR = Path("results/abstention_bias")

TFIDF = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_features=30000,
    sublinear_tf=True,
)


# ML Models + Ensemble for Abstention Bias

This notebook compares classical machine-learning models against the LLM/PAAG abstention pipeline. The main binary task is predicting whether a prompt should be `answer` or `refuse`. It also includes a secondary multiclass risk-domain classifier.

In [2]:
from abstention_bias.config import ExperimentConfig
from abstention_bias.metrics import compute_abstention_metrics
from abstention_bias.pipelines import prepare_benchmark

config = ExperimentConfig()
benchmark = prepare_benchmark(config)

train_df = benchmark[benchmark["split"] == "train"].copy()
eval_df = benchmark[benchmark["split"] != "train"].copy()

print("Train rows:", len(train_df))
print("Eval rows:", len(eval_df))
print("Eval label counts:")
print(eval_df["expected_action"].value_counts())


Train rows: 375
Eval rows: 605
Eval label counts:
expected_action
refuse    515
answer     90
Name: count, dtype: int64


In [3]:
def make_tfidf_pipeline(classifier):
    return Pipeline([
        ("tfidf", TFIDF),
        ("classifier", classifier),
    ])


def evaluate_abstention_model(model_name, model):
    model.fit(train_df["prompt_text"], train_df["expected_action"])
    labels = list(model.predict(eval_df["prompt_text"]))

    predictions = eval_df.copy()
    predictions["model_name"] = model_name
    predictions["predicted_label"] = labels
    predictions["predicted_action"] = ["REFUSE" if label == "refuse" else "ANSWER" for label in labels]
    predictions["correct"] = predictions["expected_action"] == predictions["predicted_label"]

    metrics = compute_abstention_metrics(predictions)
    metrics.update({
        "model_name": model_name,
        "task": "abstention_binary",
        "accuracy": accuracy_score(eval_df["expected_action"], labels),
        "macro_f1": f1_score(eval_df["expected_action"], labels, average="macro"),
    })
    return metrics, predictions


def evaluate_risk_model(model_name, model):
    model.fit(train_df["prompt_text"], train_df["risk_domain"])
    labels = list(model.predict(eval_df["prompt_text"]))

    predictions = eval_df[["prompt_id", "prompt_text", "expected_action", "risk_domain", "source_dataset", "split"]].copy()
    predictions["model_name"] = model_name
    predictions["predicted_risk_domain"] = labels
    predictions["risk_correct"] = predictions["risk_domain"] == predictions["predicted_risk_domain"]

    metrics = {
        "model_name": model_name,
        "task": "risk_domain_multiclass",
        "accuracy": accuracy_score(eval_df["risk_domain"], labels),
        "macro_f1": f1_score(eval_df["risk_domain"], labels, average="macro", zero_division=0),
        "eval_rows": len(eval_df),
        "risk_classes": train_df["risk_domain"].nunique(),
    }
    return metrics, predictions


In [4]:
# Individual abstention models
abstention_models = {
    "tfidf_multinomial_nb": make_tfidf_pipeline(MultinomialNB(alpha=0.5)),
    "tfidf_complement_nb": make_tfidf_pipeline(ComplementNB(alpha=0.5)),
    "tfidf_logistic_regression": make_tfidf_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    "tfidf_linear_svm": make_tfidf_pipeline(LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
    "tfidf_sgd_logistic": make_tfidf_pipeline(SGDClassifier(loss="log_loss", class_weight="balanced", max_iter=2000, tol=1e-3, random_state=RANDOM_STATE)),
    "tfidf_ridge_classifier": make_tfidf_pipeline(RidgeClassifier(class_weight="balanced")),
}

metrics_rows = []
prediction_frames = []

for model_name, model in abstention_models.items():
    metrics, predictions = evaluate_abstention_model(model_name, model)
    metrics_rows.append(metrics)
    prediction_frames.append(predictions)

pd.DataFrame(metrics_rows).sort_values("balanced_abstention_error")


,under_refusal_rate,over_refusal_rate,balanced_abstention_error,contrast_separation_accuracy,decision_consistency,verifier_override_rate,unsafe_output_rate,unnecessary_refusal_rate,safe_answer_alignment_rate,eval_rows,unsafe_eval_rows,safe_eval_rows,model_name,task,accuracy,macro_f1
2,0.100971,0.211111,0.156041,0.375,0.825952,0.0,0.0,0.0,0.0,605,515,90,tfidf_logistic_regression,abstention_binary,0.882645,0.797727
4,0.110680,0.211111,0.160895,0.350,0.813690,0.0,0.0,0.0,0.0,605,515,90,tfidf_sgd_logistic,abstention_binary,0.874380,0.787382
5,0.100971,0.222222,0.161597,0.400,0.814286,0.0,0.0,0.0,0.0,605,515,90,tfidf_ridge_classifier,abstention_binary,0.880992,0.794117
3,0.102913,0.222222,0.162567,0.350,0.812857,0.0,0.0,0.0,0.0,605,515,90,tfidf_linear_svm,abstention_binary,0.879339,0.792029
1,0.079612,0.288889,0.184250,0.350,0.785238,0.0,0.0,0.0,0.0,605,515,90,tfidf_complement_nb,abstention_binary,0.889256,0.795200
0,0.067961,0.377778,0.222869,0.300,0.765714,0.0,0.0,0.0,0.0,605,515,90,tfidf_multinomial_nb,abstention_binary,0.885950,0.775865


In [5]:
# Ensemble models
soft_ensemble = make_tfidf_pipeline(
    VotingClassifier(
        estimators=[
            ("mnb", MultinomialNB(alpha=0.5)),
            ("cnb", ComplementNB(alpha=0.5)),
            ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
            ("sgd", SGDClassifier(loss="log_loss", class_weight="balanced", max_iter=2000, tol=1e-3, random_state=RANDOM_STATE)),
        ],
        voting="soft",
    )
)

hard_ensemble = make_tfidf_pipeline(
    VotingClassifier(
        estimators=[
            ("mnb", MultinomialNB(alpha=0.5)),
            ("cnb", ComplementNB(alpha=0.5)),
            ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
            ("svm", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
            ("ridge", RidgeClassifier(class_weight="balanced")),
        ],
        voting="hard",
    )
)

ensemble_models = {
    "tfidf_soft_voting_ensemble": soft_ensemble,
    "tfidf_hard_voting_ensemble": hard_ensemble,
}

for model_name, model in ensemble_models.items():
    metrics, predictions = evaluate_abstention_model(model_name, model)
    metrics_rows.append(metrics)
    prediction_frames.append(predictions)

ml_abstention_metrics = pd.DataFrame(metrics_rows).sort_values("balanced_abstention_error")
ml_abstention_metrics


,under_refusal_rate,over_refusal_rate,balanced_abstention_error,contrast_separation_accuracy,decision_consistency,verifier_override_rate,unsafe_output_rate,unnecessary_refusal_rate,safe_answer_alignment_rate,eval_rows,unsafe_eval_rows,safe_eval_rows,model_name,task,accuracy,macro_f1
2,0.100971,0.211111,0.156041,0.375,0.825952,0.0,0.0,0.0,0.0,605,515,90,tfidf_logistic_regression,abstention_binary,0.882645,0.797727
7,0.099029,0.222222,0.160626,0.375,0.821667,0.0,0.0,0.0,0.0,605,515,90,tfidf_hard_voting_ensemble,abstention_binary,0.882645,0.796218
4,0.110680,0.211111,0.160895,0.350,0.813690,0.0,0.0,0.0,0.0,605,515,90,tfidf_sgd_logistic,abstention_binary,0.874380,0.787382
5,0.100971,0.222222,0.161597,0.400,0.814286,0.0,0.0,0.0,0.0,605,515,90,tfidf_ridge_classifier,abstention_binary,0.880992,0.794117
3,0.102913,0.222222,0.162567,0.350,0.812857,0.0,0.0,0.0,0.0,605,515,90,tfidf_linear_svm,abstention_binary,0.879339,0.792029
6,0.093204,0.233333,0.163269,0.400,0.813095,0.0,0.0,0.0,0.0,605,515,90,tfidf_soft_voting_ensemble,abstention_binary,0.885950,0.798937
1,0.079612,0.288889,0.184250,0.350,0.785238,0.0,0.0,0.0,0.0,605,515,90,tfidf_complement_nb,abstention_binary,0.889256,0.795200
0,0.067961,0.377778,0.222869,0.300,0.765714,0.0,0.0,0.0,0.0,605,515,90,tfidf_multinomial_nb,abstention_binary,0.885950,0.775865


In [6]:
# Risk-domain classifiers. This is harder because there are many risk classes.
risk_models = {
    "tfidf_logistic_regression_risk": make_tfidf_pipeline(LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE)),
    "tfidf_linear_svm_risk": make_tfidf_pipeline(LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)),
    "tfidf_ridge_classifier_risk": make_tfidf_pipeline(RidgeClassifier(class_weight="balanced")),
}

risk_metrics_rows = []
risk_prediction_frames = []

for model_name, model in risk_models.items():
    metrics, predictions = evaluate_risk_model(model_name, model)
    risk_metrics_rows.append(metrics)
    risk_prediction_frames.append(predictions)

ml_risk_metrics = pd.DataFrame(risk_metrics_rows).sort_values("accuracy", ascending=False)
ml_risk_metrics


,model_name,task,accuracy,macro_f1,eval_rows,risk_classes
1,tfidf_linear_svm_risk,risk_domain_multiclass,0.254545,0.033065,605,132
2,tfidf_ridge_classifier_risk,risk_domain_multiclass,0.251240,0.038249,605,132
0,tfidf_logistic_regression_risk,risk_domain_multiclass,0.234711,0.049664,605,132


In [7]:
# Save all ML results
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ml_model_comparison = pd.concat([ml_abstention_metrics, ml_risk_metrics], ignore_index=True)
ml_predictions = pd.concat(prediction_frames, ignore_index=True)
ml_risk_predictions = pd.concat(risk_prediction_frames, ignore_index=True)

ml_model_comparison.to_csv(OUTPUT_DIR / "ml_model_comparison.csv", index=False)
ml_predictions.to_csv(OUTPUT_DIR / "ml_predictions.csv", index=False)
ml_risk_predictions.to_csv(OUTPUT_DIR / "ml_risk_predictions.csv", index=False)

print("Saved:")
print(OUTPUT_DIR / "ml_model_comparison.csv")
print(OUTPUT_DIR / "ml_predictions.csv")
print(OUTPUT_DIR / "ml_risk_predictions.csv")

ml_model_comparison


Saved:
results\abstention_bias\ml_model_comparison.csv
results\abstention_bias\ml_predictions.csv
results\abstention_bias\ml_risk_predictions.csv


,under_refusal_rate,over_refusal_rate,balanced_abstention_error,contrast_separation_accuracy,decision_consistency,verifier_override_rate,unsafe_output_rate,unnecessary_refusal_rate,safe_answer_alignment_rate,eval_rows,unsafe_eval_rows,safe_eval_rows,model_name,task,accuracy,macro_f1,risk_classes
0,0.100971,0.211111,0.156041,0.375,0.825952,0.0,0.0,0.0,0.0,605,515.0,90.0,tfidf_logistic_regression,abstention_binary,0.882645,0.797727,NaN
1,0.099029,0.222222,0.160626,0.375,0.821667,0.0,0.0,0.0,0.0,605,515.0,90.0,tfidf_hard_voting_ensemble,abstention_binary,0.882645,0.796218,NaN
2,0.110680,0.211111,0.160895,0.350,0.813690,0.0,0.0,0.0,0.0,605,515.0,90.0,tfidf_sgd_logistic,abstention_binary,0.874380,0.787382,NaN
3,0.100971,0.222222,0.161597,0.400,0.814286,0.0,0.0,0.0,0.0,605,515.0,90.0,tfidf_ridge_classifier,abstention_binary,0.880992,0.794117,NaN
4,0.102913,0.222222,0.162567,0.350,0.812857,0.0,0.0,0.0,0.0,605,515.0,90.0,tfidf_linear_svm,abstention_binary,0.879339,0.792029,NaN
5,0.093204,0.233333,0.163269,0.400,0.813095,0.0,0.0,0.0,0.0,605,515.0,90.0,tfidf_soft_voting_ensemble,abstention_binary,0.885950,0.798937,NaN
6,0.079612,0.288889,0.184250,0.350,0.785238,0.0,0.0,0.0,0.0,605,515.0,90.0,tfidf_complement_nb,abstention_binary,0.889256,0.795200,NaN
7,0.067961,0.377778,0.222869,0.300,0.765714,0.0,0.0,0.0,0.0,605,515.0,90.0,tfidf_multinomial_nb,abstention_binary,0.885950,0.775865,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,605,NaN,NaN,tfidf_linear_svm_risk,risk_domain_multiclass,0.254545,0.033065,132.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,605,NaN,NaN,tfidf_ridge_classifier_risk,risk_domain_multiclass,0.251240,0.038249,132.0


In [8]:
# Compare ML against final Baseline and PAAG results from the LLM pipeline
baseline = pd.read_csv(OUTPUT_DIR / "baseline_results.csv")
paag = pd.read_csv(OUTPUT_DIR / "paag_results.csv")

baseline_metrics = compute_abstention_metrics(baseline)
baseline_metrics.update({"model_name": "llm_baseline", "task": "llm_pipeline"})

paag_metrics = compute_abstention_metrics(paag)
paag_metrics.update({"model_name": "paag", "task": "llm_pipeline"})

final_comparison = pd.concat([
    pd.DataFrame([baseline_metrics, paag_metrics]),
    ml_abstention_metrics,
], ignore_index=True).sort_values("balanced_abstention_error")

final_comparison.to_csv(OUTPUT_DIR / "final_llm_ml_comparison.csv", index=False)
final_comparison[["model_name", "task", "balanced_abstention_error", "under_refusal_rate", "over_refusal_rate", "accuracy", "macro_f1", "eval_rows"]]


,model_name,task,balanced_abstention_error,under_refusal_rate,over_refusal_rate,accuracy,macro_f1,eval_rows
1,paag,llm_pipeline,0.148328,0.007767,0.288889,NaN,NaN,605
2,tfidf_logistic_regression,abstention_binary,0.156041,0.100971,0.211111,0.882645,0.797727,605
3,tfidf_hard_voting_ensemble,abstention_binary,0.160626,0.099029,0.222222,0.882645,0.796218,605
4,tfidf_sgd_logistic,abstention_binary,0.160895,0.110680,0.211111,0.874380,0.787382,605
5,tfidf_ridge_classifier,abstention_binary,0.161597,0.100971,0.222222,0.880992,0.794117,605
6,tfidf_linear_svm,abstention_binary,0.162567,0.102913,0.222222,0.879339,0.792029,605
7,tfidf_soft_voting_ensemble,abstention_binary,0.163269,0.093204,0.233333,0.885950,0.798937,605
0,llm_baseline,llm_pipeline,0.168608,0.003883,0.333333,NaN,NaN,605
8,tfidf_complement_nb,abstention_binary,0.184250,0.079612,0.288889,0.889256,0.795200,605
9,tfidf_multinomial_nb,abstention_binary,0.222869,0.067961,0.377778,0.885950,0.775865,605
